In [1]:
import numpy as np

In [ ]:
def Levenberg_Marquardt(r, x0, itermax=1000, tol=1e-6, lam=1e-3, v=10.0, J=None):
    """
    Algoritmo de Levenberg-Marquardt para optimización de mínimos cuadrados no lineales.
    
    Parámetros:
    -----------
    r : función
        Función vectorial de residuos r(x).
    x0 : array-like
        Vector de estimación inicial.
    itermax : int
        Número máximo de iteraciones.
    tol : float
        Tolerancia para el criterio de parada.
    lam : float
        Parámetro de amortiguación inicial (lambda).
    v : float
        Factor multiplicador/divisor para ajustar lambda dinámicamente.
    J : función, opcional
        Función que calcula la matriz Jacobiana J(x). Si es None, se calcula
        por diferencias finitas.
        
    Retorna:
    --------
    xk : ndarray
        Aproximación de la solución del problema.
    orbita : list
        Historial de los puntos xk aceptados.
    error : float
        Norma del residuo final ||r(xk)||.
    """
    x0 = np.array(x0, dtype=float).flatten()
    xk = x0.copy()
    orbita = [xk.copy()]
    n = len(xk)
    
    # Función interna para aproximación numérica de la Jacobiana
    def jacobiano_numerico(x, eps=1e-8):
        rx = r(x)
        m = len(rx)
        J_mat = np.zeros((m, n))
        for j in range(n):
            x_step = x.copy()
            x_step[j] += eps
            J_mat[:, j] = (r(x_step) - rx) / eps
        return J_mat

    iter_count = 0
    rx = r(xk)
    cost_k = 0.5 * np.dot(rx, rx)

    while np.linalg.norm(rx) > tol and iter_count < itermax:
        # Evaluar la Jacobiana analítica o numérica
        Jx = J(xk) if J is not None else jacobiano_numerico(xk)
        
        JtJ = Jx.T @ Jx
        Jtr = Jx.T @ rx
        
        # Sistema modificado: (J^T J + lambda * I) s = -J^T r
        A = JtJ + lam * np.eye(n)
        
        try:
            sk = np.linalg.solve(A, -Jtr)
        except np.linalg.LinAlgError:
            sk, _, _, _ = np.linalg.lstsq(A, -Jtr, rcond=None)

        # Probar el nuevo punto candidato
        x_new = xk + sk
        r_new = r(x_new)
        cost_new = 0.5 * np.dot(r_new, r_new)

        # Evaluar si hubo mejora en la función de costo
        if cost_new < cost_k:
            # Aceptar paso y reducir parámetro de amortiguación lambda
            xk = x_new
            rx = r_new
            cost_k = cost_new
            lam = lam / v
            orbita.append(xk.copy())
            
            # Criterio de parada secundario basado en la magnitud del paso
            if np.linalg.norm(sk) < tol * (np.linalg.norm(xk) + 1.0):
                break
        else:
            # Rechazar paso e incrementar parámetro lambda para favorecer descenso de gradiente
            lam = lam * v

        iter_count += 1

    return xk, orbita, np.linalg.norm(rx)

In [4]:
f = lambda X: np.array([
    -400 * X[0] * (X[1] - X[0]**2) - 2 * (1 - X[0]),
    200 * (X[1] - X[0]**2)
])

x0 = np.array([0.2, 0.2])
xk, orbita, error = Gauss_Newton(f, x0, int(1e3), 1e-6)

print('Solución:', xk)
print('Error:', error)
print('Iteraciones:', len(orbita) - 1)

Solución: [1. 1.]
Error: 4.863152313964054e-08
Iteraciones: 6
